In [1]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.cluster import KMeans


In [2]:
def load_hand_landmarks(parquet_path):
    df = pd.read_parquet(parquet_path)

    # Keep only hands
    df = df[df["type"].isin(["left_hand", "right_hand"])]

    frames = sorted(df["frame"].unique())
    T = len(frames)

    if T < 2:
        return None

    hand_xyz = np.zeros((T, 42, 3), dtype=np.float32)

    for t, frame in enumerate(frames):
        fdf = df[df["frame"] == frame]

        left = fdf[fdf["type"] == "left_hand"].sort_values("landmark_index")
        right = fdf[fdf["type"] == "right_hand"].sort_values("landmark_index")

        if len(left) == 21:
            hand_xyz[t, :21] = left[["x", "y", "z"]].values

        if len(right) == 21:
            hand_xyz[t, 21:] = right[["x", "y", "z"]].values

    return hand_xyz


In [3]:
def compute_motion_score(hand_xyz):
    if hand_xyz is None or hand_xyz.shape[0] < 2:
        return 0.0

    diffs = hand_xyz[1:] - hand_xyz[:-1]
    dist = np.linalg.norm(diffs, axis=-1)

    valid = dist[~np.isnan(dist)]

    if len(valid) == 0:
        return 0.0

    return valid.mean()


In [4]:
def analyze_dataset(train_csv_path, base_path, sample_limit=None):

    train_df = pd.read_csv(train_csv_path)

    if sample_limit:
        train_df = train_df.head(sample_limit)

    motion_scores = []

    for _, row in tqdm(train_df.iterrows(), total=len(train_df)):
        parquet_path = os.path.join(base_path, row["path"])

        try:
            hand_xyz = load_hand_landmarks(parquet_path)

            if hand_xyz is None:
                motion_scores.append(0.0)
                continue

            score = compute_motion_score(hand_xyz)
            motion_scores.append(score)

        except:
            motion_scores.append(0.0)

    train_df["motion_score"] = motion_scores

    return train_df


In [5]:
def create_class_clusters(df, n_clusters=4):

    # Average motion per class
    class_motion = (
        df.groupby("sign")["motion_score"]
        .mean()
        .reset_index()
        .rename(columns={"motion_score": "avg_motion_score"})
    )

    scores = class_motion["avg_motion_score"].values.reshape(-1, 1)

    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    class_motion["motion_cluster"] = kmeans.fit_predict(scores)

    # Sort clusters from static → dynamic
    cluster_order = (
        class_motion.groupby("motion_cluster")["avg_motion_score"]
        .mean()
        .sort_values()
        .index
    )

    mapping = {old: new for new, old in enumerate(cluster_order)}
    class_motion["motion_cluster"] = class_motion["motion_cluster"].map(mapping)

    print("\nCluster meaning:")
    print(class_motion.groupby("motion_cluster")["avg_motion_score"].mean())

    return class_motion


In [6]:
def assign_clusters_to_samples(df, class_motion):

    df = df.merge(
        class_motion[["sign", "motion_cluster"]],
        on="sign",
        how="left"
    )

    return df


In [7]:
def create_subsets(df):

    subsets = {}

    for cluster_id in sorted(df["motion_cluster"].unique()):
        subset_df = df[df["motion_cluster"] == cluster_id].reset_index(drop=True)
        subsets[cluster_id] = subset_df

        print(f"Cluster {cluster_id}:")
        print("  Samples:", len(subset_df))
        print("  Classes:", subset_df["sign"].nunique())

    return subsets


In [8]:
if __name__ == "__main__":

    TRAIN_CSV = "/kaggle/input/asl-signs/train.csv"
    BASE_PATH = "/kaggle/input/asl-signs"

    # FULL dataset (no sampling)
    df = analyze_dataset(TRAIN_CSV, BASE_PATH)

    print("Finished computing motion scores.")



100%|██████████| 94477/94477 [2:21:39<00:00, 11.12it/s]

Finished computing motion scores.


In [9]:
def create_static_dynamic_split(df):
    
    # Average motion per class
    class_motion = (
        df.groupby("sign")["motion_score"]
        .mean()
        .reset_index()
        .rename(columns={"motion_score": "avg_motion_score"})
    )

    # Median threshold (balanced split)
    threshold = class_motion["avg_motion_score"].median()

    class_motion["motion_category"] = class_motion["avg_motion_score"].apply(
        lambda x: "static" if x <= threshold else "dynamic"
    )

    return class_motion


In [10]:
def assign_category_to_samples(df, class_motion):

    df = df.merge(
        class_motion[["sign", "motion_category"]],
        on="sign",
        how="left"
    )

    return df


In [11]:
class_motion = create_static_dynamic_split(df)

df = assign_category_to_samples(df, class_motion)

print("Samples per category:")
print(df["motion_category"].value_counts())

print("\nClasses per category:")
print(df.groupby("motion_category")["sign"].nunique())


Samples per category:
motion_category
static     47446
dynamic    47031
Name: count, dtype: int64

Classes per category:
motion_category
dynamic    125
static     125
Name: sign, dtype: int64


In [12]:
static_df = df[df["motion_category"] == "static"].reset_index(drop=True)
dynamic_df = df[df["motion_category"] == "dynamic"].reset_index(drop=True)

print("Static dataset:", static_df.shape)
print("Dynamic dataset:", dynamic_df.shape)


Static dataset: (47446, 6)
Dynamic dataset: (47031, 6)


In [13]:
static_df.to_csv("train_static.csv", index=False)
dynamic_df.to_csv("train_dynamic.csv", index=False)
